# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**What this section asks:** say what a single row means, then state the time period represented by its measurements.

**Contract:** one row is one pseudonymized content item (page) in a snapshot. The activity totals and derived rates describe a trailing 90-day window ending at export time. The comparison fields divide part of that history into the most recent 30 days (`*_last_30d`) and the 30 days before them (`*_prev_30d`). This CSV has no per-row calendar date, so we cannot calculate exact `MIN(date)` / `MAX(date)` here; the window definition comes from the data dictionary.

**Why the next cell exists:** it loads the local starter CSV, prints only aggregate counts (never the pseudonymous IDs), and checks that `content_id` really is unique. Zero duplicate IDs supports the one-row-per-content claim.

In [10]:
from pathlib import Path
import pandas as pd

# Find the repo root even if Jupyter started in work/notebooks.
repo_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "data/raw/content_refresh_anonymized.csv").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv from this notebook's folder.")

csv_path = repo_root / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)

# Grain check: a content item should appear once in this starter snapshot.
duplicate_content_ids = df["content_id"].duplicated().sum()

print(f"Dataset: {csv_path.relative_to(repo_root)}")
print(f"Rows: {len(df):,}; columns: {len(df.columns)}")
print(f"Distinct clients: {df['client_id'].nunique():,}")
print(f"Rows with duplicate content_id: {duplicate_content_ids:,}")
print("Documented window: trailing 90-day snapshot; comparison fields split into two 30-day windows.")

assert len(df) == 30_000, "Row count differs from the starter data dictionary; check which file was loaded."
assert len(df.columns) == 44, "Column count differs from the starter data dictionary."
assert df["client_id"].nunique() == 32, "Client count differs from the starter data dictionary."
assert duplicate_content_ids == 0, "The claimed one-row-per-content grain does not hold."

Dataset: data\raw\content_refresh_anonymized.csv
Rows: 30,000; columns: 44
Distinct clients: 32
Rows with duplicate content_id: 0
Documented window: trailing 90-day snapshot; comparison fields split into two 30-day windows.


## 2. Fields: feature / label / context / excluded

**What this section asks:** decide which columns could be used before the decision, which define the outcome, which are just identifiers/context, and which must not enter the model. Each source column must land in exactly one bucket.

**Contract for this starter exercise:** the outcome is a *current-trend proxy*: `is_declining_label = (trend_direction == "down")`. For a leakage-aware test, candidate features are content metadata plus the previous 30-day measurements. The most recent 30-day impressions define the outcome period, while the 90-day aggregates and their rates include that period, so keep them out of this model's feature inputs. This is a teaching contract for the starter snapshot, not a claim of future prediction. A future-outcome capstone needs fresh warehouse windows: past-only features, then a later target window.

- **Feature candidates:** metadata and previous-window values that could be known at the start of the most-recent 30-day outcome window. Confirm availability at the real decision point before modeling.
- **Label / proxy:** the field defining the proxy and the current-period outcome used to interpret it.
- **Context:** pseudonymous IDs for grouping, joining, and client-held-out splitting only.
- **Excluded:** overlapping-window measurements, their derived buckets/rates, and generation metadata explicitly marked as non-features. The reason is listed by group in the code.

**Why the next cell exists:** it checks that all 44 source columns are covered once, validates the observed trend categories, and constructs the proxy label from its documented rule.

In [11]:
# Put every raw column in exactly one bucket. These are starter-contract decisions,
# not a universal feature list; the timing of a future project can change them.
feature_candidates = [
    "search_volume", "competition", "competition_level", "cpc", "content_type", "main_intent",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "age_tier", "age_tier_order", "freshness_tier", "word_count_tier", "char_count_tier",
]
label_proxy_fields = ["trend_direction", "trend_pct", "impressions_last_30d"]
context_fields = ["content_id", "client_id"]
excluded_fields = [
    # Generation metadata is documented as not a model feature.
    "provider_used", "model_used",
    # These 90-day totals overlap the current 30-day outcome period.
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    # These current-period values are not prior-only features.
    "clicks_last_30d", "sessions_last_30d",
    # Current-window rates/position and their tiers overlap the outcome period.
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier",
]

field_buckets = {
    "feature candidates": feature_candidates,
    "label / proxy": label_proxy_fields,
    "context": context_fields,
    "excluded": excluded_fields,
}
all_bucketed_fields = [field for fields in field_buckets.values() for field in fields]

# Catch a missing column, misspelling, or accidental double-classification.
assert len(all_bucketed_fields) == len(set(all_bucketed_fields)), "A field appears in more than one bucket."
assert set(all_bucketed_fields) == set(df.columns), (
    f"Bucket mismatch. Missing: {sorted(set(df.columns) - set(all_bucketed_fields))}; "
    f"unknown: {sorted(set(all_bucketed_fields) - set(df.columns))}"
)

for bucket_name, fields in field_buckets.items():
    print(f"{bucket_name} ({len(fields)}): {', '.join(fields)}")

# The CSV has no separate target column, so construct the documented proxy here.
allowed_directions = {"new", "flat", "up", "down", "stable"}
unexpected_directions = set(df["trend_direction"].dropna().unique()) - allowed_directions
assert not unexpected_directions, f"Unexpected trend categories: {unexpected_directions}"
decline_proxy = df["trend_direction"].eq("down").astype("int8")
assert set(decline_proxy.unique()) <= {0, 1}
print(f"Proxy positives: {decline_proxy.sum():,} / {len(decline_proxy):,} ({decline_proxy.mean():.1%})")
print("Proxy created: 1 means trend_direction is 'down'; this is not a future-outcome label.")

feature candidates (18): search_volume, competition, competition_level, cpc, content_type, main_intent, word_count, char_count, content_age_days, days_since_last_update, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, age_tier, age_tier_order, freshness_tier, word_count_tier, char_count_tier
label / proxy (3): trend_direction, trend_pct, impressions_last_30d
context (2): content_id, client_id
excluded (21): provider_used, model_used, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, clicks_last_30d, sessions_last_30d, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, impression_tier, position_tier
Proxy positives: 16,262 / 30,000 (54.2%)
Proxy created: 1 means trend_direction is 'down'; this is not a future-outcome label.


## 3. Verify it with queries (grain, counts, missing values, windows)

**What this section asks:** show evidence for the contract rather than relying on the dictionary alone. Check where values are missing, whether the gaps follow content type, and whether the documented rate formulas agree with the raw counts.

**Time-window note:** this is a snapshot with aggregate fields, not a dated daily table. We can verify the comparison-window arithmetic, but cannot calculate calendar `MIN(date)` / `MAX(date)` from this CSV. The exact 90-day and 30-day definitions are documented in the data dictionary.

**Why the next cell exists:** it prints compact missingness summaries, checks missingness by `content_type`, and compares the stored CTR/trend percentages to their documented formulas. It never prints content or client IDs.

In [12]:
import numpy as np

# Overall missingness: sort high-to-low so the largest gaps are easy to spot.
missing_rate = df.isna().mean().sort_values(ascending=False)
print("Highest missing fractions:")
print(missing_rate.head(12).map(lambda value: f"{value:.1%}").to_string())

# A grouped check tells us whether blanks are concentrated in a content category.
missing_by_type = df.groupby("content_type", dropna=False)[
    ["search_volume", "competition", "cpc", "main_intent", "word_count"]
].agg(lambda values: values.isna().mean())
print("\nMissing fraction by content_type:")
print(missing_by_type.round(3).to_string())

# These counts help interpret two documented special cases.
previous_window_zero = df["impressions_prev_30d"].eq(0)
print(f"\nRows with zero previous-window impressions: {previous_window_zero.sum():,}")
print(f"Rows with missing trend_pct: {df['trend_pct'].isna().sum():,}")
print(f"Rows where avg_position is zero (no position data): {df['avg_position'].eq(0).sum():,}")

# Verify CTR = clicks / impressions * 100 (stored to two decimal places).
ctr_expected = df["clicks_90d"] / df["impressions_90d"] * 100
ctr_difference = (df["ctr"] - ctr_expected).abs()
max_ctr_difference = ctr_difference.max()
print(f"\nLargest absolute CTR formula difference: {max_ctr_difference:.4f} percentage points")
assert max_ctr_difference <= 0.011, "CTR differs from its documented formula by more than rounding allows."

# Verify trend_pct = (last 30d - previous 30d) / previous 30d * 100.
# Rows with zero previous impressions have no defined percentage change.
has_previous_impressions = df["impressions_prev_30d"].gt(0)
trend_expected = (
    (df.loc[has_previous_impressions, "impressions_last_30d"]
     - df.loc[has_previous_impressions, "impressions_prev_30d"])
    / df.loc[has_previous_impressions, "impressions_prev_30d"]
    * 100
)
trend_difference = (df.loc[has_previous_impressions, "trend_pct"] - trend_expected).abs()
max_trend_difference = trend_difference.max()
print(f"Largest absolute trend formula difference: {max_trend_difference:.4f} percentage points")
assert max_trend_difference <= 0.11, "trend_pct differs from its documented formula by more than rounding allows."

print("\nWindow contract verified: previous 30-day counts precede last 30-day counts; 90-day totals are overlapping context, not safe features for this proxy.")

Highest missing fractions:
provider_used        71.5%
word_count           25.7%
char_count           25.7%
word_count_tier      25.7%
char_count_tier      25.7%
model_used           19.1%
trend_pct            11.3%
competition_level     8.7%
search_volume         8.2%
cpc                   8.2%
competition           8.2%
main_intent           7.9%

Missing fraction by content_type:
                    search_volume  competition    cpc  main_intent  word_count
content_type                                                                  
comparison article          0.000        0.000  0.000         0.00       0.000
feedly article              1.000        1.000  1.000         1.00       0.000
keyword article             0.014        0.014  0.014         0.01       0.283

Rows with zero previous-window impressions: 3,388
Rows with missing trend_pct: 3,388
Rows where avg_position is zero (no position data): 1,205

Largest absolute CTR formula difference: 0.0050 percentage points
Largest 

## 4. Data limits

**What this section asks:** state the boundaries of what these data can support. A good contract prevents later analysis from quietly making stronger claims than the data allow.

**Limits for this starter snapshot:**

- It is one row per content item, not a daily panel, and its columns are aggregated windows rather than dated observations.
- `trend_direction` is a rule-derived current-trend proxy, not an independently observed future outcome. A score based on it is not proof that pages will decline or recover.
- The 90-day totals overlap the most-recent 30-day outcome window. Do not use those totals or current-window rates to claim future prediction for this proxy.
- Missingness is patterned by `content_type`; zero and missing do not always mean the same thing. In particular, `avg_position == 0` means no position data, and some rate columns can exceed 100 because their measurement systems differ.
- `content_id` and `client_id` are pseudonyms: use them for grouping/splitting, never as learned features. Do not print or publish row-level IDs.
- The data are observational. They cannot prove that refreshing a page causes recovery.

**Human-facing output:** a ranked, decision-support list of content items for an editor to review; it is not a guarantee that editing will improve performance.

**Why the next cell exists:** it checks a few of these limits against the loaded file and prints a compact contract summary without exposing row-level data.

In [13]:
# Check that this starter CSV is an aggregate snapshot, not a dated panel.
calendar_date_columns = [
    column for column in df.columns
    if column.lower() in {"date", "report_date", "export_date"}
    or pd.api.types.is_datetime64_any_dtype(df[column])
]
print(f"Calendar-date fields in starter CSV: {calendar_date_columns}")
assert not calendar_date_columns, "Unexpected calendar date found; revisit the stated time-window contract."

# These values are documented exceptions, not automatic data errors.
zero_position_rows = int(df["avg_position"].eq(0).sum())
scroll_over_100_rows = int(df["scroll_rate"].gt(100).sum())
ai_rate_over_100_rows = int(df["ai_traffic_pct"].gt(100).sum())
print(f"avg_position == 0 rows (no position data): {zero_position_rows:,}")
print(f"scroll_rate > 100 rows: {scroll_over_100_rows:,}")
print(f"ai_traffic_pct > 100 rows: {ai_rate_over_100_rows:,}")

# IDs are retained only for grouping/splitting; do not display their values.
print(f"Pseudonymous grouping columns available: {['content_id', 'client_id']}")
print("Model feature rule: exclude both ID columns; use client_id only to hold out whole clients.")

contract_summary = {
    "unit": "one pseudonymized content item per row",
    "snapshot": "trailing 90-day aggregate with recent/previous 30-day comparison fields",
    "target": "current trend proxy: trend_direction == 'down'",
    "output": "ranked decision-support list for human review",
}
for item, statement in contract_summary.items():
    print(f"{item.title()}: {statement}")

print("\nContract checks completed. See the markdown above for the interpretation and limitations.")

Calendar-date fields in starter CSV: []
avg_position == 0 rows (no position data): 1,205
scroll_rate > 100 rows: 119
ai_traffic_pct > 100 rows: 23
Pseudonymous grouping columns available: ['content_id', 'client_id']
Model feature rule: exclude both ID columns; use client_id only to hold out whole clients.
Unit: one pseudonymized content item per row
Snapshot: trailing 90-day aggregate with recent/previous 30-day comparison fields
Target: current trend proxy: trend_direction == 'down'
Output: ranked decision-support list for human review

Contract checks completed. See the markdown above for the interpretation and limitations.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section is filled — markdown thinking and code checks are included.
- [x] The notebook runs top to bottom with no errors (verified in this workspace).
- [x] No client names, URLs, or private queries are included; row-level IDs are not printed.
- [x] Claims use careful words: observed, measured, directional, decision-support.
- [ ] Committed to my repo under `work/notebooks/` and submitted my repo URL on the assignment card. (Do this after reviewing the notebook.)